# TP Machine Learning supervisé — Breast Cancer Wisconsin
## Comparaison de modèles de classification

**Problématique :** prédire si une tumeur est bénigne ou maligne à partir de caractéristiques numériques calculées à partir d'images de cellules.

**Dataset :** Breast Cancer Wisconsin Diagnostic, disponible dans Scikit-learn et décrit par l'UCI Machine Learning Repository.

**Objectifs :**
- explorer et comprendre les données ;
- construire un prétraitement sans fuite de données ;
- comparer plusieurs modèles avec une validation croisée ;
- évaluer le modèle final sur un jeu de test tenu à l'écart ;
- interpréter les résultats et formuler une conclusion critique.

> **Important :** ce notebook est un travail pédagogique. Il ne constitue pas un outil de diagnostic médical.

## 1. Importation des bibliothèques

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

## 2. Chargement du dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

# Scikit-learn code la cible 0 = malin et 1 = bénin.
# Pour faciliter l'interprétation, on crée une cible explicite :
# 1 = malin, 0 = bénin.
X = df.drop(columns=["target"])
y = (df["target"] == 0).astype(int)
y.name = "malin"

df_analyse = X.copy()
df_analyse["malin"] = y

print("Nombre de lignes :", df_analyse.shape[0])
print("Nombre de variables explicatives :", X.shape[1])
print("Noms des classes : 0 = bénin ; 1 = malin")
display(df_analyse.head())

## 3. Compréhension générale des données

In [ ]:
print("Dimensions :", df_analyse.shape)
print("\nTypes des variables :")
display(df_analyse.dtypes.value_counts().rename_axis("Type").to_frame("Nombre"))
print("\nStatistiques descriptives :")
display(X.describe().T)
print("\nValeurs manquantes par colonne (20 premières) :")
display(X.isna().sum().sort_values(ascending=False).head(20).to_frame("Valeurs manquantes"))
print("\nNombre de doublons :", X.duplicated().sum())

### 3.1 Répartition des classes

In [ ]:
class_counts = y.map({0: "Bénin", 1: "Malin"}).value_counts()
display(class_counts.to_frame("Nombre"))

ax = class_counts.plot(kind="bar", rot=0)
ax.set_title("Répartition des classes")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'observations")
plt.tight_layout()
plt.show()

### 3.2 Distribution de quelques variables

In [ ]:
variables_a_visualiser = [
    "mean radius", "mean texture", "mean perimeter",
    "mean area", "mean smoothness"
]

for col in variables_a_visualiser:
    plt.figure(figsize=(7, 4))
    for label, nom in [(0, "Bénin"), (1, "Malin")]:
        plt.hist(X.loc[y == label, col], bins=20, alpha=0.55, label=nom)
    plt.title(f"Distribution de {col} selon la classe")
    plt.xlabel(col)
    plt.ylabel("Fréquence")
    plt.legend()
    plt.tight_layout()
    plt.show()

### 3.3 Corrélations entre quelques variables

In [ ]:
cols_corr = ["mean radius", "mean texture", "mean perimeter", "mean area",
             "mean smoothness", "mean concavity"]
corr = X[cols_corr].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Corrélation")
plt.xticks(range(len(cols_corr)), cols_corr, rotation=45, ha="right")
plt.yticks(range(len(cols_corr)), cols_corr)
plt.title("Corrélations entre variables sélectionnées")
plt.tight_layout()
plt.show()

## 4. Préparation des données

In [ ]:
# Vérification de la cible et des dimensions
assert X.shape[0] == y.shape[0]
assert y.nunique() == 2

# Séparation stratifiée : le jeu de test reste à l'écart pendant le choix du modèle.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Entraînement :", X_train.shape)
print("Test :", X_test.shape)
print("\nProportions de la classe maligne :")
print("Train :", round(y_train.mean(), 3))
print("Test  :", round(y_test.mean(), 3))

In [ ]:
pretraitement_standardise = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="median")),
    ("standardisation", StandardScaler())
])

pretraitement_sans_standardisation = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="median"))
])

## 5. Définition des modèles

In [ ]:
modeles = {
    "Régression logistique": Pipeline([
        ("pretraitement", pretraitement_standardise),
        ("modele", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("pretraitement", pretraitement_standardise),
        ("modele", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Arbre de décision": Pipeline([
        ("pretraitement", pretraitement_sans_standardisation),
        ("modele", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("pretraitement", pretraitement_sans_standardisation),
        ("modele", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))
    ]),
    "SVM": Pipeline([
        ("pretraitement", pretraitement_standardise),
        ("modele", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}

print("Modèles à comparer :", ", ".join(modeles.keys()))

## 6. Comparaison par validation croisée

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

resultats_cv = []
for nom, modele in modeles.items():
    scores = cross_validate(
        modele, X_train, y_train,
        cv=cv, scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    ligne = {"Modèle": nom}
    for metrique in scoring:
        ligne[f"{metrique}_moyenne"] = scores[f"test_{metrique}"].mean()
        ligne[f"{metrique}_ecart_type"] = scores[f"test_{metrique}"].std()
    resultats_cv.append(ligne)

table_cv = pd.DataFrame(resultats_cv).sort_values("recall_moyenne", ascending=False)
display(table_cv.round(3))

### Visualisation des résultats de validation croisée

In [ ]:
metriques_a_tracer = ["accuracy_moyenne", "precision_moyenne",
                      "recall_moyenne", "f1_moyenne", "roc_auc_moyenne"]
ax = table_cv.set_index("Modèle")[metriques_a_tracer].plot(kind="bar", figsize=(12, 6))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score moyen en validation croisée")
ax.set_title("Comparaison des modèles (jeu d'entraînement)")
plt.xticks(rotation=25, ha="right")
plt.legend(["Accuracy", "Précision", "Rappel", "F1-score", "ROC-AUC"])
plt.tight_layout()
plt.show()

## 7. Choix et optimisation du modèle

Pour cette démonstration, on choisit le modèle avec le meilleur **rappel moyen en validation croisée**. Ce critère est cohérent avec l'objectif de limiter les tumeurs malignes manquées, mais il ne doit pas être utilisé seul : précision, F1-score, ROC-AUC et contexte d'utilisation comptent également.

La recherche d'hyperparamètres ci-dessous porte uniquement sur le jeu d'entraînement.

In [ ]:
nom_selectionne = table_cv.iloc[0]["Modèle"]
print("Modèle sélectionné selon le rappel CV :", nom_selectionne)

grilles = {
    "Régression logistique": {
        "modele__C": [0.01, 0.1, 1, 10]
    },
    "KNN": {
        "modele__n_neighbors": [3, 5, 7, 9, 11],
        "modele__weights": ["uniform", "distance"]
    },
    "Arbre de décision": {
        "modele__max_depth": [None, 3, 5, 8, 12],
        "modele__min_samples_leaf": [1, 2, 5]
    },
    "Random Forest": {
        "modele__max_depth": [None, 5, 10, 15],
        "modele__min_samples_leaf": [1, 2, 4],
        "modele__max_features": ["sqrt", 0.5]
    },
    "SVM": {
        "modele__C": [0.1, 1, 10],
        "modele__kernel": ["linear", "rbf"],
        "modele__gamma": ["scale", "auto"]
    }
}

recherche = GridSearchCV(
    estimator=modeles[nom_selectionne],
    param_grid=grilles[nom_selectionne],
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    refit=True
)
recherche.fit(X_train, y_train)

modele_final = recherche.best_estimator_
print("Meilleurs paramètres :", recherche.best_params_)
print("Meilleur rappel CV :", round(recherche.best_score_, 3))

## 8. Évaluation finale sur le jeu de test

Le jeu de test n'a pas servi à choisir le modèle ni ses hyperparamètres. Il est utilisé maintenant pour une estimation finale, plus honnête, des performances sur des données non vues.

In [ ]:
y_pred = modele_final.predict(X_test)
y_prob = modele_final.predict_proba(X_test)[:, 1]

resultats_test = pd.DataFrame([{
    "Modèle final": nom_selectionne,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Précision (malin)": precision_score(y_test, y_pred, zero_division=0),
    "Rappel (malin)": recall_score(y_test, y_pred, zero_division=0),
    "F1-score (malin)": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}])
display(resultats_test.round(3))

print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Bénin", "Malin"],
    zero_division=0
))

### 8.1 Matrice de confusion

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=[0, 1],
    display_labels=["Bénin", "Malin"],
    cmap="Blues"
)
plt.title("Matrice de confusion — jeu de test")
plt.tight_layout()
plt.show()

**Comment lire la matrice :**
- Un vrai bénin prédit bénin est une prédiction correcte.
- Un vrai malin prédit malin est une détection correcte.
- Un vrai malin prédit bénin est un faux négatif : c'est une erreur particulièrement préoccupante dans ce contexte.
- Un vrai bénin prédit malin est un faux positif : il peut entraîner des examens supplémentaires.



### 8.2 Courbe ROC

In [ ]:
RocCurveDisplay.from_estimator(modele_final, X_test, y_test)
plt.plot([0, 1], [0, 1], linestyle="--", label="Classement aléatoire")
plt.title("Courbe ROC — modèle final")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Comparaison avec un modèle de référence

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_baseline = baseline.predict(X_test)

print("Accuracy du modèle de référence :", round(accuracy_score(y_test, y_baseline), 3))
print("Rappel malin du modèle de référence :", round(recall_score(y_test, y_baseline, zero_division=0), 3))
print("Accuracy du modèle final :", round(accuracy_score(y_test, y_pred), 3))
print("Rappel malin du modèle final :", round(recall_score(y_test, y_pred, zero_division=0), 3))

Le modèle de référence prédit toujours la classe majoritaire. Le modèle final doit être évalué par rapport à cette référence, notamment sur le rappel et le F1-score, pas uniquement sur l'accuracy.

## 10. Conclusion critique — à compléter après exécution

In [ ]:
print("Modèle retenu :", nom_selectionne)
print("Paramètres retenus :", recherche.best_params_)
print("\nScores finaux sur le jeu de test :")
display(resultats_test.round(3))

###  conclusion 

Dans ce projet, nous avons appliqué un workflow de Machine Learning supervisé au dataset Breast Cancer Wisconsin Diagnostic. Après une exploration des données, nous avons préparé les variables et utilisé des pipelines afin de limiter les risques de fuite de données.

Nous avons comparé cinq algorithmes : régression logistique, KNN, arbre de décision, Random Forest et SVM. La comparaison a été réalisée avec une validation croisée stratifiée, en examinant l'accuracy, la précision, le rappel, le F1-score et la ROC-AUC.

